# Module 13: Distance Ranger

## Mission Briefing

So far, your Pico has sensed light, temperature, and button presses. But what about knowing how far away something is — without touching it?

Today you add **ultrasonic vision** to your Pico. A tiny sensor sends out a sound so high-pitched that no human can hear it, then listens for the echo. From that echo, your Pico can calculate distance down to the centimeter.

```
                   40 kHz pulse
    +----------+   ))))))))))   +--------+
    |  HC-SR04 |   )))))))))) > | Object |
    | Ultrasonic|  ))))))))))   |  (wall |
    |  Sensor  |               | hand,  |
    |          |   ((((((((((  | book)  |
    |          | < ((((((((((  +--------+
    +----+-----+   echo returns
         |
    +----+-----+
    |  Pico 2W |
    +----------+
```

Bats do this in total darkness. Your parking sensor does it every time you reverse. Now your Pico will too!

---
## Science Spotlight: Ultrasonic Waves & Echoes

Bats navigate in total darkness by making high-pitched sounds and listening for echoes. Your ultrasonic sensor does the exact same thing — too high-pitched for humans to hear!

The sensor sends out a burst of sound at **40,000 Hz** (40 kHz) — way above the 20 kHz limit of human hearing. It measures how long the echo takes to come back, and from that we can figure out the distance.

*Your instructor will explain how sound travels, how we calculate distance from echo time, and why we divide by 2.*

---
## Meet the HC-SR04 Ultrasonic Sensor

The HC-SR04 has two "eyes" — one is a **transmitter** (sends the ultrasonic pulse) and the other is a **receiver** (listens for the echo). It has 4 pins:

```
    +-------------------------------+
    |  (O)  HC-SR04  (O)           |
    |  T                  R        |
    | (transmitter)  (receiver)    |
    +----+----+----+----+----------+
         |    |    |    |
        VCC  Trig Echo  GND
         |    |    |    |
       5V  GP15  GP14  GND
```

- **VCC** = power (needs 5V, not 3.3V!)
- **Trig** = trigger — we send a short pulse here to start a measurement
- **Echo** = the sensor pulls this HIGH for the duration of the echo
- **GND** = ground

**How it works:** We send a 10-microsecond pulse on Trig. The sensor blasts out 8 ultrasonic pulses, then raises the Echo pin. The Echo pin stays HIGH until the echo returns. We time how long Echo is HIGH — that's our round-trip time!

---
## Wiring: Ultrasonic Sensor + Buzzer

### Circuit

```
   HC-SR04 Sensor:
   Pico VBUS (5V, pin 40) ──── wire ──── HC-SR04 VCC
   Pico GND  (pin 38)     ──── wire ──── HC-SR04 GND
   Pico GP15 (pin 20)     ──── wire ──── HC-SR04 Trig
   Pico GP14 (pin 19)     ──── wire ──── HC-SR04 Echo

   Buzzer:
   Pico GP13 (pin 17)     ──── wire ──── Buzzer (+) ──── Buzzer (-) ──── GND
```

### Step-by-Step

1. **Place the HC-SR04** on the breadboard — insert the 4 pins into 4 separate rows
2. **Connect a wire** from **VBUS (5V)** on the Pico to the **VCC** pin of the sensor
3. **Connect a wire** from **GND** on the Pico to the **GND** pin of the sensor
4. **Connect a wire** from **GP15** on the Pico to the **Trig** pin
5. **Connect a wire** from **GP14** on the Pico to the **Echo** pin
6. **Connect the buzzer** from **GP13** to **GND** (positive leg to GP13)

**Important:** The HC-SR04 needs **5V power** — use VBUS, not 3.3V!

Plug in your USB cable.

---
## Code Along: Your First Distance Reading

Let's make the sensor take one distance measurement. The process is:
1. Pull Trig LOW for 2 microseconds (reset)
2. Pull Trig HIGH for 10 microseconds (trigger the pulse)
3. Pull Trig LOW again
4. Measure how long Echo stays HIGH
5. Calculate distance!

Fill in the blanks:

In [ ]:
from machine import Pin, time_pulse_us
import time

trigger = Pin(_____, Pin.OUT)    # Which GPIO pin is Trig connected to?
echo = Pin(_____, Pin.IN)        # Which GPIO pin is Echo connected to?

# Send the trigger pulse
trigger.value(0)                 # Start LOW
time.sleep_us(2)                 # Wait 2 microseconds
trigger.value(_____)             # Go HIGH to trigger
time.sleep_us(10)                # Hold for 10 microseconds
trigger.value(0)                 # Back to LOW

# Measure how long Echo stays HIGH (timeout: 30000 us = ~5 meters)
duration = time_pulse_us(echo, 1, 30000)

# Calculate distance in centimeters
distance = duration * 0.0343 / _____    # Why do we divide by this number?

print("Duration:", duration, "us")
print("Distance:", distance, "cm")

Put your hand about 20 cm in front of the sensor and run the code. You should see a distance close to 20 cm!

**Why divide by 2?** The sound has to travel TO the object AND back. The `duration` is the **round-trip** time. We only want the one-way distance.

**What's 0.0343?** Sound travels at about 343 meters per second in air. That's 0.0343 cm per microsecond. So: distance = time (us) x 0.0343 (cm/us) / 2.

---
## Code Along: Continuous Distance Monitor

One reading is cool, but let's make it measure distance **continuously** — like a real range finder!

We'll put the trigger-measure-calculate code into a function so we can call it over and over.

Fill in the blanks:

In [ ]:
from machine import Pin, time_pulse_us
import time

trigger = Pin(15, Pin.OUT)
echo = Pin(14, Pin.IN)

def measure_distance():
    """Send an ultrasonic pulse and return distance in cm."""
    trigger.value(0)
    time.sleep_us(2)
    trigger.value(1)
    time.sleep_us(_____)
    trigger.value(0)
    
    duration = time_pulse_us(echo, _____, 30000)    # Wait for Echo to go HIGH
    
    if duration < 0:          # Timeout — no echo received
        return -1
    
    distance = duration * 0.0343 / 2
    return distance

# Continuously print distance
while True:
    dist = _____()            # Call our measurement function
    if dist < 0:
        print("Out of range!")
    else:
        print("Distance:", round(dist, 1), "cm")
    time.sleep(0.3)

Move your hand closer and farther from the sensor. Watch the numbers change in real time!

**Note:** If you see "Out of range!" it means the echo took too long (object too far or at a weird angle). The HC-SR04 works best between about **2 cm and 400 cm**.

---
## Code Along: Parking Sensor

Ever heard the beeping when a car backs up near something? That's a parking sensor — it beeps **faster** as the car gets closer to an obstacle. Let's build one!

The idea: measure the distance, then beep the buzzer. The closer the object, the shorter the delay between beeps.

Fill in the blanks:

In [ ]:
from machine import Pin, PWM, time_pulse_us
import time

trigger = Pin(15, Pin.OUT)
echo = Pin(14, Pin.IN)
buzzer = PWM(Pin(_____))         # Which GPIO pin is the buzzer on?
buzzer.freq(2000)                # Set buzzer tone to 2000 Hz

def measure_distance():
    trigger.value(0)
    time.sleep_us(2)
    trigger.value(1)
    time.sleep_us(10)
    trigger.value(0)
    duration = time_pulse_us(echo, 1, 30000)
    if duration < 0:
        return -1
    return duration * 0.0343 / 2

while True:
    dist = measure_distance()
    
    if dist < 0 or dist > 100:
        # Too far or out of range — stay silent
        buzzer.duty_u16(0)
        time.sleep(0.5)
    elif dist < _____:           # DANGER ZONE — very close! What distance in cm?
        # Continuous tone (object is very close)
        buzzer.duty_u16(5000)
        time.sleep(0.1)
    else:
        # Beep! Closer = shorter delay between beeps
        buzzer.duty_u16(5000)
        time.sleep(0.05)
        buzzer.duty_u16(_____)
        # Map distance (10-100 cm) to delay (0.05-0.5 seconds)
        delay = dist / 100 * 0.5
        time.sleep(delay)
    
    print("Distance:", round(dist, 1), "cm")

Move your hand toward the sensor:
- **Far away (>100 cm):** No beeping
- **Getting closer:** Slow beeps
- **Very close (<10 cm):** Continuous tone — watch out!

Congratulations — you just built a parking sensor!

---
## Experiments

Try these and write down what you observe:

1. **Measure to a wall:** Point the sensor at a wall. Walk toward it slowly. What's the closest distance it can measure accurately? (This is the **minimum range**.)

2. **Maximum range:** Point the sensor down a long hallway. What's the farthest distance it can read before showing "Out of range"?

3. **Angles matter:** Hold a book flat in front of the sensor. Now tilt the book at a 45-degree angle. What happens to the reading? Why?

4. **Soft vs hard surfaces:** Try measuring distance to a pillow, a book, and a metal water bottle. Which gives the most accurate readings? Which is hardest to detect?

5. **Speed of sound:** The formula uses 343 m/s for the speed of sound. This is at room temperature (~20 degrees C). Sound actually travels faster in warmer air. If your classroom is warmer or cooler, how might that affect your readings?

---
## Challenge: Proximity LED Bar Graph

Build a **visual distance indicator** using LEDs! Remember the 3 LEDs from Module 3 (Blink and Beyond)? Wire up 3 LEDs and make them show distance like a bar graph:

- **Far (>60 cm):** No LEDs on
- **Medium (30-60 cm):** 1 LED on (green)
- **Close (15-30 cm):** 2 LEDs on (green + yellow)
- **Very close (<15 cm):** All 3 LEDs on (green + yellow + red) + buzzer!

### Hints:
- You already know how to control multiple LEDs from Module 3
- Use `if/elif/else` to check distance ranges
- Each LED needs its own GPIO pin and 220-ohm resistor
- Combine the parking sensor buzzer with the LED bar graph for the full effect!

```
   Suggested LED wiring:
   GP16 ──── [220 ohm] ──── LED 1 (green)  ──── GND
   GP17 ──── [220 ohm] ──── LED 2 (yellow) ──── GND
   GP18 ──── [220 ohm] ──── LED 3 (red)    ──── GND
```

Get creative! Ask your instructor if you get stuck.

---
## Vocabulary Recap

| Word | What It Means |
|------|---------------|
| **Ultrasonic** | Sound above human hearing — higher than 20,000 Hz (20 kHz) |
| **HC-SR04** | An ultrasonic distance sensor with a transmitter and receiver |
| **Trigger** | A short pulse we send to tell the sensor to start measuring |
| **Echo** | The signal that comes back after the ultrasonic pulse bounces off an object |
| **time_pulse_us** | A MicroPython function that measures how long a pin stays HIGH, in microseconds |
| **Speed of sound** | About 343 meters per second in air at room temperature |
| **Round-trip** | The sound travels to the object AND back — so we divide by 2 |
| **Range** | The minimum and maximum distances the sensor can measure accurately (~2-400 cm) |

---
*Circuit Explorers — Module 13: Distance Ranger*